# All-NBA Team Prediction using Random Forest

Predicting which players make All-NBA teams (1st, 2nd, 3rd) using NBA player statistics from 2011-2025.

## 1. Data Loading & Initial Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', 60)
sns.set_style('whitegrid')

In [ ]:
# Load the master dataset with all features and player names
df_full = pd.read_csv('data/combined/all_years_combined_with_lebron.csv')
print(f'Shape: {df_full.shape}')
df_full.head()

In [ ]:
df_full.info()

In [ ]:
df_full.describe()

## 2. Target Variable Analysis

In [ ]:
# Target distribution (0=None, 1=NBA 3rd Team, 2=NBA 2nd Team, 3=NBA 1st Team)
print('All_NBA_Target value counts:')
print(df_full['All_NBA_Target'].value_counts().sort_index())
print()
print('All_NBA_Label value counts:')
print(df_full['All_NBA_Label'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of target distribution
target_counts = df_full['All_NBA_Target'].value_counts().sort_index()
labels = ['Not All-NBA', '3rd Team', '2nd Team', '1st Team']
colors = ['#95a5a6', '#e67e22', '#3498db', '#e74c3c']
axes[0].bar(labels, target_counts.values, color=colors)
axes[0].set_title('All-NBA Target Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# All-NBA selections by year
allnba_by_year = df_full[df_full['All_NBA_Target'] > 0].groupby('Season_End_Year')['All_NBA_Target'].count()
axes[1].bar(allnba_by_year.index, allnba_by_year.values, color='#3498db')
axes[1].set_title('All-NBA Selections per Season')
axes[1].set_xlabel('Season End Year')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3. Feature Exploration

In [ ]:
# Missing values
missing = df_full.isnull().sum()
missing_pct = (missing / len(df_full) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Compare key stats: All-NBA vs Non All-NBA players
key_stats = ['PTS', 'TRB', 'AST', 'STL', 'BLK', 'PER', 'WS', 'BPM', 'VORP', 'LEBRON', 'WAR', 'MP', 'G']
comparison = df_full.groupby('All_NBA_Target')[key_stats].mean().round(2)
comparison.index = ['Not All-NBA', '3rd Team', '2nd Team', '1st Team'][:len(comparison)]
comparison.T

In [ ]:
# Box plots of key features by All-NBA status
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
plot_features = ['PTS', 'PER', 'WS', 'BPM', 'VORP', 'WAR']

for ax, feat in zip(axes.ravel(), plot_features):
    df_full.boxplot(column=feat, by='All_NBA_Target', ax=ax)
    ax.set_title(feat)
    ax.set_xlabel('All-NBA Target')

plt.suptitle('Feature Distributions by All-NBA Status', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Position distribution among All-NBA selections
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pos_allnba = df_full[df_full['All_NBA_Target'] > 0]['Pos'].value_counts()
pos_all = df_full['Pos'].value_counts()

axes[0].bar(pos_all.index, pos_all.values, color='#95a5a6', alpha=0.7)
axes[0].set_title('Position Distribution (All Players)')
axes[0].set_ylabel('Count')

axes[1].bar(pos_allnba.index, pos_allnba.values, color='#e74c3c', alpha=0.7)
axes[1].set_title('Position Distribution (All-NBA Players)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numeric features with target
numeric_cols = df_full.select_dtypes(include=[np.number]).columns
correlations = df_full[numeric_cols].corr()['All_NBA_Target'].drop('All_NBA_Target').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 12))
correlations.plot(kind='barh', ax=ax, color=correlations.apply(lambda x: '#e74c3c' if x > 0 else '#3498db'))
ax.set_title('Feature Correlations with All-NBA Target')
ax.set_xlabel('Correlation')
plt.tight_layout()
plt.show()

## 4. Data Preparation for Modeling

In [ ]:
# Load the hybrid trimmed dataset (balanced mix of basic + advanced + impact features)
df = pd.read_csv('data/model_datasets/hybrid_trimmed_dataset.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
# Encode the position column
le_pos = LabelEncoder()
df['Pos_encoded'] = le_pos.fit_transform(df['Pos'].astype(str))
print('Position encoding:', dict(zip(le_pos.classes_, le_pos.transform(le_pos.classes_))))

# Define features and target
drop_cols = ['Pos', 'All_NBA_Target']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df['All_NBA_Target'].copy()

# Handle missing values - fill with 0 (e.g., 3P% for players with 0 attempts)
print(f'\nMissing values before fill:\n{X.isnull().sum()[X.isnull().sum() > 0]}')
X = X.fillna(0)

print(f'\nFeatures ({len(feature_cols)}): {feature_cols}')
print(f'Target distribution:\n{y.value_counts().sort_index()}')

In [ ]:
# Train/test split - use Season_End_Year for temporal split
# Train on 2011-2022, test on 2023-2025 (more realistic evaluation)
train_mask = X['Season_End_Year'] <= 2022
test_mask = X['Season_End_Year'] > 2022

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f'Train set: {X_train.shape[0]} samples ({y_train.value_counts().sort_index().to_dict()})')
print(f'Test set:  {X_test.shape[0]} samples ({y_test.value_counts().sort_index().to_dict()})')

## 5. Random Forest Model

In [ ]:
# Train Random Forest with class_weight='balanced' to handle imbalance
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
print('Training complete.')
print(f'Train accuracy: {rf.score(X_train, y_train):.4f}')
print(f'Test accuracy:  {rf.score(X_test, y_test):.4f}')

In [ ]:
# Classification report
y_pred = rf.predict(X_test)
target_names = ['Not All-NBA', '3rd Team', '2nd Team', '1st Team']
# Only include labels that exist in test set
present_labels = sorted(y_test.unique())
present_names = [target_names[i] for i in present_labels]

print(classification_report(y_test, y_pred, labels=present_labels, target_names=present_names))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred, labels=present_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=present_names)
disp.plot(ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix - Random Forest')
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
# Feature importance
importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
importance.plot(kind='barh', ax=ax, color='#3498db')
ax.set_title('Random Forest Feature Importance')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 7. Cross-Validation

In [ ]:
# Stratified K-Fold cross-validation on the full dataset
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X, y, cv=skf, scoring='accuracy')
print(f'Cross-validation accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
print(f'Individual fold scores: {cv_scores.round(4)}')

## 8. Predictions - Who Would Make All-NBA?

In [ ]:
# Load full dataset with player names to see who the model predicts
df_names = pd.read_csv('data/combined/all_years_combined_with_lebron.csv')

# Get test set predictions with player names
test_players = df_names[df_names['Season_End_Year'] > 2022][['Player', 'Team', 'Pos', 'Season_End_Year', 'All_NBA_Label', 'All_NBA_Target']].reset_index(drop=True)
test_players['Predicted'] = y_pred
test_players['Predicted_Label'] = test_players['Predicted'].map({0: 'None', 1: 'NBA3', 2: 'NBA2', 3: 'NBA1'})

# Show predicted All-NBA players
predicted_allnba = test_players[test_players['Predicted'] > 0].sort_values(['Season_End_Year', 'Predicted'], ascending=[True, False])
print(f'Predicted All-NBA selections ({len(predicted_allnba)} players):')
predicted_allnba

In [ ]:
# Compare predictions vs actual for test years
for year in sorted(test_players['Season_End_Year'].unique()):
    year_data = test_players[test_players['Season_End_Year'] == year]
    print(f'\n{"=" * 50}')
    print(f'Season {year}')
    print(f'{"=" * 50}')
    
    actual = year_data[year_data['All_NBA_Target'] > 0][['Player', 'Team', 'Pos', 'All_NBA_Label']]
    predicted = year_data[year_data['Predicted'] > 0][['Player', 'Team', 'Pos', 'Predicted_Label']]
    
    print(f'\nActual All-NBA ({len(actual)}):')
    if len(actual) > 0:
        print(actual.to_string(index=False))
    else:
        print('  (no data)')
    
    print(f'\nPredicted All-NBA ({len(predicted)}):')
    if len(predicted) > 0:
        print(predicted.to_string(index=False))
    else:
        print('  (none predicted)')

## 9. Binary Classification: All-NBA or Not

The multiclass model above struggles to distinguish between 1st/2nd/3rd team (only ~60 samples each, and the distinction is subjective voting margins). A binary model — **All-NBA vs Not** — is a more robust formulation.

**Issues addressed from the multiclass model:**
- **Misleading accuracy**: 96.4% baseline by predicting all 0s. We now focus on F1, precision-recall curves.
- **Missing value handling**: Use median instead of 0 for `3P%` (0 implies 0% shooter, not "no attempts").
- **Better evaluation**: Precision-recall curve, probability calibration, and per-year breakdown.

In [ ]:
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    f1_score, PrecisionRecallDisplay
)

# Reload and prepare data
df_bin = pd.read_csv('data/model_datasets/hybrid_trimmed_dataset.csv')

# Binary target: 1 if any All-NBA team, 0 otherwise
df_bin['All_NBA_Binary'] = (df_bin['All_NBA_Target'] > 0).astype(int)

print('Binary target distribution:')
print(df_bin['All_NBA_Binary'].value_counts())
print(f'\nPositive rate: {df_bin["All_NBA_Binary"].mean():.3%}')

In [ ]:
# Encode position
le_pos_bin = LabelEncoder()
df_bin['Pos_encoded'] = le_pos_bin.fit_transform(df_bin['Pos'].astype(str))

# Features - drop original target, label, and string Pos
drop_cols_bin = ['Pos', 'All_NBA_Target', 'All_NBA_Binary']
feature_cols_bin = [c for c in df_bin.columns if c not in drop_cols_bin]

X_bin = df_bin[feature_cols_bin].copy()
y_bin = df_bin['All_NBA_Binary'].copy()

# Better missing value handling: median fill instead of 0
for col in X_bin.columns:
    if X_bin[col].isnull().any():
        median_val = X_bin[col].median()
        print(f'Filling {col} NaN ({X_bin[col].isnull().sum()} values) with median={median_val:.3f}')
        X_bin[col] = X_bin[col].fillna(median_val)

# Temporal split: train 2011-2022, test 2023-2025
train_mask_bin = X_bin['Season_End_Year'] <= 2022
test_mask_bin = X_bin['Season_End_Year'] > 2022

X_train_bin, X_test_bin = X_bin[train_mask_bin], X_bin[test_mask_bin]
y_train_bin, y_test_bin = y_bin[train_mask_bin], y_bin[test_mask_bin]

print(f'\nTrain: {X_train_bin.shape[0]} ({y_train_bin.sum()} positives, {y_train_bin.mean():.2%})')
print(f'Test:  {X_test_bin.shape[0]} ({y_test_bin.sum()} positives, {y_test_bin.mean():.2%})')
print(f'\nBaseline accuracy (predict all 0): {1 - y_test_bin.mean():.3%}')

In [ ]:
# Train binary Random Forest
rf_bin = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_bin.fit(X_train_bin, y_train_bin)

y_pred_bin = rf_bin.predict(X_test_bin)
y_proba_bin = rf_bin.predict_proba(X_test_bin)[:, 1]

print('Binary Random Forest Results')
print('=' * 50)
print(f'Test accuracy:  {rf_bin.score(X_test_bin, y_test_bin):.4f}')
print(f'Baseline (all 0): {1 - y_test_bin.mean():.4f}')
print(f'\nF1 Score: {f1_score(y_test_bin, y_pred_bin):.4f}')
print(f'Average Precision (PR-AUC): {average_precision_score(y_test_bin, y_proba_bin):.4f}')
print()
print(classification_report(y_test_bin, y_pred_bin, target_names=['Not All-NBA', 'All-NBA']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm_bin = confusion_matrix(y_test_bin, y_pred_bin)
ConfusionMatrixDisplay(cm_bin, display_labels=['Not All-NBA', 'All-NBA']).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Confusion Matrix - Binary RF')

# Precision-Recall curve (much more informative than ROC for imbalanced data)
precision, recall, thresholds = precision_recall_curve(y_test_bin, y_proba_bin)
ap = average_precision_score(y_test_bin, y_proba_bin)
axes[1].plot(recall, precision, color='#e74c3c', lw=2, label=f'RF (AP={ap:.3f})')
axes[1].axhline(y=y_test_bin.mean(), color='gray', linestyle='--', label=f'Baseline ({y_test_bin.mean():.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].set_xlim([0, 1.05])
axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance for binary model
importance_bin = pd.Series(rf_bin.feature_importances_, index=feature_cols_bin).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
importance_bin.plot(kind='barh', ax=ax, color='#3498db')
ax.set_title('Feature Importance - Binary Random Forest')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation with F1 scoring (more meaningful than accuracy for imbalanced data)
skf_bin = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_acc = cross_val_score(rf_bin, X_bin, y_bin, cv=skf_bin, scoring='accuracy')
cv_f1 = cross_val_score(rf_bin, X_bin, y_bin, cv=skf_bin, scoring='f1')
cv_ap = cross_val_score(rf_bin, X_bin, y_bin, cv=skf_bin, scoring='average_precision')

print('5-Fold Cross-Validation Results')
print('=' * 50)
print(f'Accuracy:          {cv_acc.mean():.4f} (+/- {cv_acc.std():.4f})')
print(f'F1 Score:          {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})')
print(f'Avg Precision (AP): {cv_ap.mean():.4f} (+/- {cv_ap.std():.4f})')

In [ ]:
# Per-year predictions with player names and probability scores
df_names = pd.read_csv('data/combined/all_years_combined_with_lebron.csv')
test_players_bin = df_names[df_names['Season_End_Year'] > 2022][['Player', 'Team', 'Pos', 'Season_End_Year', 'All_NBA_Label', 'All_NBA_Target']].reset_index(drop=True)
test_players_bin['Actual'] = (test_players_bin['All_NBA_Target'] > 0).astype(int)
test_players_bin['Predicted'] = y_pred_bin
test_players_bin['Probability'] = y_proba_bin

for year in sorted(test_players_bin['Season_End_Year'].unique()):
    year_data = test_players_bin[test_players_bin['Season_End_Year'] == year].copy()
    
    print(f'\n{"=" * 60}')
    print(f'Season {year}')
    print(f'{"=" * 60}')
    
    # Show top 20 by probability
    top = year_data.nlargest(20, 'Probability')[['Player', 'Team', 'Pos', 'Actual', 'Predicted', 'Probability', 'All_NBA_Label']]
    top['Probability'] = top['Probability'].round(3)
    top['Correct'] = (top['Actual'] == top['Predicted']).map({True: 'Y', False: ''})
    print(top.to_string(index=False))
    
    # Summary stats
    actual_pos = year_data['Actual'].sum()
    pred_pos = year_data['Predicted'].sum()
    correct = ((year_data['Actual'] == 1) & (year_data['Predicted'] == 1)).sum()
    print(f'\nActual: {actual_pos} | Predicted: {pred_pos} | Correct: {correct}/{actual_pos}')